In [1]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules/python-utils")
sys.path.append("/workspaces/dev/modules/ai-utils")

In [2]:
import librosa
import numpy as np
from pathlib import Path
from IPython.display import Audio

In [3]:
from sj_utils.file.yaml import load_yaml
from sj_utils.collection import SafetyDict
from sj_utils.audio import segment_audio, load_audio_from_mp4

In [4]:
from rt_whisper import streamers, saveloaders
from rt_whisper.data import Param, TokenState

/workspaces/dev/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 42
SAMPLE_RATE = 16000
HYPERPARAMETER = "/workspaces/dev/test/performance_test/esic/hyperparameters/20250730/step1_16b-RTX3090/trial_wer3o8_2123_20250730_154355.yaml"
# SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/test-clean/61/70968/61-70968-0001.flac"
SOURCE = "/workspaces/dev/.datasets/ESIC-v1.1/v1.1/dev/20090203/014_017_EN_Ždanoka/en.OS.man-diar.mp4"
LOG_FILE_PATH = "/workspaces/dev/logs/core.log"

In [6]:
src = Path(SOURCE)
src.exists()
log = Path(LOG_FILE_PATH)
if log.exists():
    with log.open("w"): pass
hyperparameter = Path(HYPERPARAMETER)

In [7]:
np.random.seed(SEED)
audio, sr = load_audio_from_mp4(src, sr=SAMPLE_RATE)
# audio, sr = librosa.load(src, sr=SAMPLE_RATE)
segments = segment_audio(audio)
Audio(audio, rate=sr)

/workspaces/dev/modules/python-utils/sj_utils/audio.py:42: WavFileWarning: Reached EOF prematurely; finished at 2499661 bytes, expected 4294967303 bytes from header.
  sample_rate, waveform = wavfile.read(io.BytesIO(out))


In [8]:
hyperparameter = SafetyDict(load_yaml(hyperparameter)[1])

In [ ]:
# SAVED_PATH = "/workspaces/dev/storage/esic/112000/dev/20090203/014_017_EN_Ždanoka/en.OS.man-diar"
# saved_path = Path(SAVED_PATH)
# token_streamer = saveloaders.get_token_streamer_loader(saved_path, hyperparameter)

In [9]:
token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter(hyperparameter=hyperparameter)
token_streamer = streamers.get_token_streamer_with_vad_v2_min_filter()

Key 'max_prompt_words' not found in SafetyDict. Returning default value.


In [10]:
param = Param()
completed = []
index = -1

In [ ]:
from rt_whisper.processors.asr.data import ASRState

index += 1
chunk = segments[index]

param.chunk = chunk
param.language = "en"
ctx:TokenState = token_streamer.process(param, get_context = True)
result = ctx.extract()
completed.extend(result.completed)
param.update(result, update_prompt=True)

print(f"{index}" + "--" * 20)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.completed]
)
print(
    [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
    for v in result.candidate]
)

In [ ]:
Audio(result.context_dict[ASRState].chunk, rate=sr)

In [11]:
param = Param()
completed = []
for i, segment in enumerate(segments):
    param.chunk = segment
    param.language = "en"
    ctx:TokenState = token_streamer.process(param, get_context = True)
    result = ctx.extract()
    completed.extend(result.completed)
    param.update(result, update_prompt=True)

completed.extend(result.candidate)

TypeError: '>' not supported between instances of 'int' and 'NoneType'

In [ ]:
for s in completed:
    print(s.lang, s.text)